In [ ]:
!pip install pandas openpyxl scikit-learn xgboost matplotlib

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import matplotlib.pyplot as plt

# 1. 載入資料
import sys
from pathlib import Path
_root = Path.cwd().resolve()
for _ in range(10):
    if (_root / "src" / "data_layout.py").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root / "src"))
from data_layout import resolve_processed_data_dir
_csv_dir = resolve_processed_data_dir(_root)
df = pd.read_csv(_csv_dir / "2025_metrics.csv")
if "Season" in df.columns:
    df = df[df["Season"] == 2025].reset_index(drop=True)

# 3. 準備資料
target = 'Win Rate'
exclude_cols = [c for c in ['Team', 'Season', target] if c in df.columns]
X = df.drop(columns=exclude_cols)
y = df[target]

# 4. 標準化特徵
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# 5. 訓練 XGBoost 模型
model = xgb.XGBRegressor(n_estimators=100, random_state=42)
model.fit(X_scaled, y)

# 6. 取得特徵重要性（按 gain）
importance_dict = model.get_booster().get_score(importance_type='gain')
importance = pd.Series(importance_dict).sort_values(ascending=False)

# 7. 印出前幾個重要特徵
print("Top Features by Gain:")
print(importance.head(15))

# 8. 畫圖（可選）
importance.plot(kind='barh', title="XGBoost Feature Importance (Gain)")
plt.xlabel("Mean Gain")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()
